<a href="https://colab.research.google.com/github/brenonsx066-ship-it/Dr.-comunnity/blob/main/controller.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
from datetime import datetime
from model import conexao, cursor, atendimento_service
from view import (
    exibir_cabecalho,
    pausar,
    obter_data,
    validar_data,
    validar_horario,
    mostrar_medicos
)

def cadastrar_paciente():
    exibir_cabecalho("CADASTRO DE PACIENTE")

    while True:
        nome = input("\nNome completo: ").strip()

        if nome and all(
            parte.isalpha() for parte in nome.split()
        ):
            break

        print("Digite um nome válido, sem números.")

    while True:
        senha = input("Crie uma senha: ").strip()

        if len(senha) >= 4:
            break

        print("A senha deve ter pelo menos 4 caracteres.")

    while True:
        try:
            idade = int(input("Idade: "))

            if idade >= 0:
                break

            print("A idade não pode ser negativa.")

        except ValueError:
            print("Digite uma idade válida.")

    while True:
        queixa = input("Queixa ou sintoma: ").strip()

        if queixa:
            break

        print("A queixa não pode ficar vazia.")

    while True:
        try:
            dor = int(input("Nível de dor de 0 a 10: "))

            if 0 <= dor <= 10:
                break

            print("Digite um valor de 0 a 10.")

        except ValueError:
            print("Digite somente números inteiros.")

    status = "Atenção" if dor >= 7 else "Registrado"

    try:
        atendimento_service.cadastrar_paciente(
            nome, senha, idade, queixa, dor, status
        )

        print("\nCadastro realizado com sucesso!")
        print(f"Bem-vindo(a), {nome}!")

    except sqlite3.IntegrityError:
        print("\nJá existe um cadastro com esse nome.")

    pausar()

def realizar_login():
    exibir_cabecalho("LOGIN DO PACIENTE")

    nome = input("Nome cadastrado: ").strip()
    senha = input("Senha: ").strip()

    cursor.execute("""
    SELECT id, nome, idade, queixa, niveldedor, status
    FROM pacientes
    WHERE LOWER(nome) = LOWER(?) AND senha = ?
    """, (nome, senha))

    paciente = cursor.fetchone()

    if paciente is None:
        print("\nNome ou senha incorretos.")
        pausar()
        return

    acessar_menu_paciente(paciente)

def acessar_menu_paciente(paciente):

    while True:
        (
            paciente_id, nome, idade,
            queixa, dor, status
        ) = paciente

        exibir_cabecalho("DR. COMMUNITY - TELA INICIAL")

        print(f"\nOlá, {nome}!")
        print("Acompanhe sua saúde e seus atendimentos.")

        print("\nRESUMO DA SAÚDE")
        print(f"Idade: {idade}")
        print(f"Queixa: {queixa}")
        print(f"Nível de dor informado: {dor}/10")
        print(f"Status registrado: {status}")

        print("\nACESSO RÁPIDO")
        print("[1] Agendar consulta")
        print("[2] Resultado dos exames")
        print("[3] Guia médico")
        print("[4] Bate-papo")
        print("[5] Meus dados")
        print("[6] Agendamentos realizados")
        print("[7] Notificações")
        print("[8] Ajuda e acessibilidade")
        print("[9] Assistente virtual")
        print("[10] Editar queixa e nível de dor")
        print("[0] Sair da conta")

        opcao = input("\nEscolha uma opção: ").strip()

        if opcao == "1":
            agendar_consulta(paciente_id)

        elif opcao == "2":
            consultar_exames(paciente_id)

        elif opcao == "3":
            guia_medico()

        elif opcao == "4":
            menu_bate_papo(paciente_id)

        elif opcao == "5":
            meus_dados(paciente_id)

        elif opcao == "6":
            meus_agendamentos(paciente_id)

        elif opcao == "7":
            consultar_notificacoes(paciente_id)

        elif opcao == "8":
            menu_acessibilidade()

        elif opcao == "9":
            assistente_virtual()

        elif opcao == "10":
            editar_dados_saude(paciente_id)

        elif opcao == "0":
            print("\nVocê saiu da conta.")
            pausar()
            break

        else:
            print("\nOpção inválida.")
            pausar()

        cursor.execute("""
        SELECT id, nome, idade, queixa, niveldedor, status
        FROM pacientes
        WHERE id = ?
        """, (paciente_id,))

        paciente_atualizado = cursor.fetchone()

        if paciente_atualizado is None:
            break

        paciente = paciente_atualizado

def guia_medico():
    while True:
        exibir_cabecalho("GUIA MÉDICO")

        print("Encontre um profissional de saúde.")
        print("\n[1] Busca rápida")
        print("[2] Busca detalhada")
        print("[3] Voltar")

        opcao = input("\nEscolha: ").strip()

        if opcao == "1":
            busca_rapida_medico()

        elif opcao == "2":
            busca_detalhada_medico()

        elif opcao == "3":
            break

        else:
            print("Opção inválida.")
            pausar()

def busca_rapida_medico():
    exibir_cabecalho("BUSCA RÁPIDA")

    termo = input(
        "Digite o nome do médico ou especialidade: "
    ).strip()

    cursor.execute("""
    SELECT id, nome, especialidade
    FROM medicos
    WHERE nome LIKE ? OR especialidade LIKE ?
    ORDER BY nome
    """, (f"%{termo}%", f"%{termo}%"))

    mostrar_medicos(cursor.fetchall())
    pausar()

def busca_detalhada_medico():
    exibir_cabecalho("BUSCA DETALHADA")

    nome = input(
        "Nome do médico (ENTER para ignorar): "
    ).strip()

    especialidade = input(
        "Especialidade (ENTER para ignorar): "
    ).strip()

    cursor.execute("""
    SELECT id, nome, especialidade
    FROM medicos
    WHERE nome LIKE ?
    AND especialidade LIKE ?
    ORDER BY nome
    """, (f"%{nome}%", f"%{especialidade}%"))

    mostrar_medicos(cursor.fetchall())
    pausar()

def agendar_consulta(paciente_id):
    exibir_cabecalho("AGENDAMENTO ONLINE")

    cursor.execute("""
    SELECT id, nome, especialidade
    FROM medicos
    ORDER BY nome
    """)

    medicos = cursor.fetchall()

    if not medicos:
        print("\nNão existem médicos cadastrados.")
        print("O agendamento depende do cadastro dos profissionais.")
        pausar()
        return

    mostrar_medicos(medicos)

    try:
        medico_id = int(
            input("\nDigite o ID do médico escolhido: ")
        )
    except ValueError:
        print("ID inválido.")
        pausar()
        return

    cursor.execute("""
    SELECT id, nome, especialidade
    FROM medicos
    WHERE id = ?
    """, (medico_id,))

    medico = cursor.fetchone()

    if medico is None:
        print("Médico não encontrado.")
        pausar()
        return

    _, nome_medico, especialidade = medico

    data = input("Data da consulta (DD/MM/AAAA): ").strip()

    if not validar_data(data):
        print("Data inválida. Use DD/MM/AAAA.")
        pausar()
        return

    horario = input("Horário (HH:MM): ").strip()

    if not validar_horario(horario):
        print("Horário inválido. Use HH:MM.")
        pausar()
        return

    print("\n[1] Presencial")
    print("[2] Online")

    tipo_opcao = input("Tipo de atendimento: ").strip()

    if tipo_opcao == "1":
        tipo = "Presencial"
    elif tipo_opcao == "2":
        tipo = "Online"
    else:
        print("Tipo de atendimento inválido.")
        pausar()
        return

    cursor.execute("""
    SELECT id FROM agendamentos
    WHERE medico_id = ?
    AND data = ?
    AND horario = ?
    AND status = 'Confirmado'
    """, (medico_id, data, horario))

    if cursor.fetchone():
        print("\nEsse horário já está confirmado para o médico.")
        pausar()
        return

    cursor.execute("""
    INSERT INTO agendamentos
    (paciente_id, medico_id, medico, especialidade,
     data, horario, tipo, status)
    VALUES (?, ?, ?, ?, ?, ?, ?, 'Pendente')
    """, (
        paciente_id, medico_id, nome_medico,
        especialidade, data, horario, tipo
    ))

    conexao.commit()

    print("\nSolicitação registrada!")
    print(f"Médico: {nome_medico}")
    print(f"Data: {data}")
    print(f"Horário: {horario}")
    print("Situação: Pendente de confirmação.")

    pausar()

def meus_agendamentos(paciente_id):
    while True:
        exibir_cabecalho("AGENDAMENTOS REALIZADOS")

        cursor.execute("""
        SELECT id, medico, especialidade, data,
               horario, tipo, status
        FROM agendamentos
        WHERE paciente_id = ?
        ORDER BY id DESC
        """, (paciente_id,))

        consultas = cursor.fetchall()

        if not consultas:
            print("\nVocê ainda não possui consultas.")
        else:
            for consulta in consultas:
                (
                    consulta_id, medico, especialidade,
                    data, horario, tipo, status
                ) = consulta

                print(f"\nAgendamento ID: {consulta_id}")
                print(f"Médico: {medico}")
                print(f"Especialidade: {especialidade}")
                print(f"Data: {data}")
                print(f"Horário: {horario}")
                print(f"Tipo: {tipo}")
                print(f"Situação: {status}")
                print("-" * 45)

        print("\n[1] Cancelar consulta")
        print("[2] Voltar")

        opcao = input("Escolha: ").strip()

        if opcao == "1":
            cancelar_agendamento(paciente_id)

        elif opcao == "2":
            break

        else:
            print("Opção inválida.")
            pausar()

def cancelar_agendamento(paciente_id):
    exibir_cabecalho("CANCELAR CONSULTA")

    try:
        consulta_id = int(
            input("ID do agendamento que deseja cancelar: ")
        )
    except ValueError:
        print("Digite um ID numérico.")
        pausar()
        return

    cursor.execute("""
    SELECT status
    FROM agendamentos
    WHERE id = ? AND paciente_id = ?
    """, (consulta_id, paciente_id))

    consulta = cursor.fetchone()

    if consulta is None:
        print("\nAgendamento não encontrado.")
        pausar()
        return

    if consulta[0] in ("Cancelado", "Recusado"):
        print("\nEsse agendamento não está ativo.")
        pausar()
        return

    cursor.execute("""
    UPDATE agendamentos
    SET status = 'Cancelado'
    WHERE id = ? AND paciente_id = ?
    """, (consulta_id, paciente_id))

    conexao.commit()

    print("\nAgendamento cancelado.")
    pausar()

def consultar_exames(paciente_id):
    exibir_cabecalho("RESULTADOS DOS EXAMES")

    cursor.execute("""
    SELECT id, nome, data, resultado, observacao
    FROM exames
    WHERE paciente_id = ?
    ORDER BY id DESC
    """, (paciente_id,))

    exames = cursor.fetchall()

    if not exames:
        print("\nNenhum resultado de exame foi disponibilizado.")
        print("Consulte a equipe de saúde sobre seus exames.")

    else:
        for exame in exames:
            exame_id, nome, data, resultado, observacao = exame

            print(f"\nExame ID: {exame_id}")
            print(f"Nome: {nome}")
            print(f"Data: {data}")
            print(f"Resultado: {resultado}")
            print(f"Observação: {observacao}")
            print("-" * 45)

    pausar()

def menu_bate_papo(paciente_id):
    while True:
        exibir_cabecalho("BATE-PAPO")

        print("[1] Enviar mensagem")
        print("[2] Ver minhas mensagens")
        print("[3] Voltar")

        opcao = input("\nEscolha: ").strip()

        if opcao == "1":
            enviar_mensagem(paciente_id)

        elif opcao == "2":
            ver_mensagens(paciente_id)

        elif opcao == "3":
            break

        else:
            print("Opção inválida.")
            pausar()

def enviar_mensagem(paciente_id):
    exibir_cabecalho("ENVIAR MENSAGEM")

    cursor.execute("""
    SELECT id, nome, especialidade
    FROM medicos
    ORDER BY nome
    """)

    medicos = cursor.fetchall()

    if not medicos:
        print("\nNenhum médico cadastrado.")
        pausar()
        return

    mostrar_medicos(medicos)

    try:
        medico_id = int(
            input("\nID do médico (0 para mensagem geral): ")
        )
    except ValueError:
        print("ID inválido.")
        pausar()
        return

    if medico_id == 0:
        medico_id = None
    else:
        cursor.execute(
            "SELECT id FROM medicos WHERE id = ?",
            (medico_id,)
        )

        if cursor.fetchone() is None:
            print("Médico não encontrado.")
            pausar()
            return

    mensagem = input("\nDigite sua mensagem: ").strip()

    if not mensagem:
        print("A mensagem não pode ficar vazia.")
        pausar()
        return

    cursor.execute("""
    INSERT INTO mensagens
    (paciente_id, medico_id, remetente, mensagem, data_envio)
    VALUES (?, ?, 'Paciente', ?, ?)
    """, (
        paciente_id, medico_id, mensagem, obter_data()
    ))

    conexao.commit()

    print("\nMensagem salva com sucesso.")
    print("O envio para o médico exige a integração")
    print("com o painel do profissional.")

    pausar()

def ver_mensagens(paciente_id):
    exibir_cabecalho("MINHAS MENSAGENS")

    cursor.execute("""
    SELECT mensagens.remetente,
           medicos.nome,
           mensagens.mensagem,
           mensagens.data_envio
    FROM mensagens
    LEFT JOIN medicos ON mensagens.medico_id = medicos.id
    WHERE mensagens.paciente_id = ?
    ORDER BY mensagens.id
    """, (paciente_id,))

    mensagens = cursor.fetchall()

    if not mensagens:
        print("\nVocê ainda não possui mensagens.")

    else:
        for remetente, medico, mensagem, data in mensagens:
            destino = medico if medico else "Atendimento geral"

            print(f"\nRemetente: {remetente}")
            print(f"Contato: {destino}")
            print(f"Data: {data}")
            print(f"Mensagem: {mensagem}")
            print("-" * 45)

    pausar()

def consultar_notificacoes(paciente_id):
    exibir_cabecalho("NOTIFICAÇÕES")

    cursor.execute("""
    SELECT medico, data, horario, status
    FROM agendamentos
    WHERE paciente_id = ?
    ORDER BY id DESC
    """, (paciente_id,))

    consultas = cursor.fetchall()

    if not consultas:
        print("\nVocê não possui notificações de consultas.")
    else:
        for medico, data, horario, status in consultas:
            print(f"\nConsulta com: {medico}")
            print(f"Data: {data}, às {horario}")
            print(f"Situação: {status}")

    print("\nEssas informações são consultadas no banco.")
    print("Notificações automáticas ainda precisam")
    print("ser integradas ao sistema.")

    pausar()

def menu_acessibilidade():
    exibir_cabecalho("AJUDA E ACESSIBILIDADE")

    print("[1] Como navegar")
    print("[2] Recursos para pessoas surdas")
    print("[3] Voltar")

    opcao = input("\nEscolha: ").strip()

    if opcao == "1":
        print("\nDigite o número da função desejada.")
        print("As informações importantes aparecem por escrito.")

    elif opcao == "2":
        print("\nRecursos planejados para o Dr. Community:")
        print("- Comunicação por texto.")
        print("- Mensagens e notificações visuais.")
        print("- Instruções claras.")
        print("- Possível integração com Libras.")
        print("- Possível transcrição de fala para texto.")

        print("\nAs integrações avançadas ainda precisam")
        print("ser desenvolvidas.")

    elif opcao == "3":
        return

    else:
        print("Opção inválida.")

    pausar()

def assistente_virtual():
    exibir_cabecalho("ASSISTENTE VIRTUAL")

    print("Olá! Posso ajudar com informações gerais.")
    print("Não consigo diagnosticar doenças.")
    print("Descreva sua dúvida ou sintoma.\n")

    pergunta = input("Você: ").strip().lower()

    if not pergunta:
        print("\nDigite uma pergunta para continuar.")

    elif any(
        termo in pergunta
        for termo in [
            "não consigo respirar",
            "falta de ar intensa",
            "dor forte no peito",
            "desmaiei",
            "convulsão"
        ]
    ):
        print("\nATENÇÃO")
        print("Esses sintomas podem exigir atendimento imediato.")
        print("Procure um serviço de emergência.")
        print("No Brasil, ligue 192 (SAMU) em uma emergência.")

    elif any(
        termo in pergunta
        for termo in [
            "dor", "febre", "sintoma",
            "enjoo", "tontura", "mal-estar"
        ]
    ):
        print("\nOrientação geral:")
        print("Os sintomas podem ter diversas causas.")
        print("Não é possível determinar a causa apenas")
        print("por esta conversa.")
        print("Procure um profissional de saúde para avaliação.")
        print("Se os sintomas forem intensos ou piorarem,")
        print("busque atendimento prontamente.")

    else:
        print("\nPosso fornecer informações gerais de saúde,")
        print("mas não substituo a avaliação de um profissional.")
        print("Para dúvidas específicas, procure a equipe de saúde.")

    pausar()

def meus_dados(paciente_id):
    exibir_cabecalho("MEUS DADOS")

    cursor.execute("""
    SELECT nome, idade, queixa, niveldedor, status
    FROM pacientes
    WHERE id = ?
    """, (paciente_id,))

    paciente = cursor.fetchone()

    if paciente:
        nome, idade, queixa, dor, status = paciente

        print(f"\nNome: {nome}")
        print(f"Idade: {idade}")
        print(f"Queixa: {queixa}")
        print(f"Nível de dor informado: {dor}/10")
        print(f"Status: {status}")

    else:
        print("Dados não encontrados.")

    pausar()

def editar_dados_saude(paciente_id):
    exibir_cabecalho("EDITAR DADOS DE SAÚDE")

    queixa = input("Nova queixa ou sintoma: ").strip()

    if not queixa:
        print("A queixa não pode ficar vazia.")
        pausar()
        return

    while True:
        try:
            dor = int(input("Novo nível de dor (0 a 10): "))

            if 0 <= dor <= 10:
                break

            print("Digite um valor entre 0 e 10.")

        except ValueError:
            print("Digite um número inteiro.")

    status = "Atenção" if dor >= 7 else "Registrado"

    cursor.execute("""
    UPDATE pacientes
    SET queixa = ?, niveldedor = ?, status = ?
    WHERE id = ?
    """, (queixa, dor, status, paciente_id))

    conexao.commit()

    print("\nDados atualizados.")
    print("Esta classificação é apenas um registro simples,")
    print("não uma avaliação médica.")

    pausar()

def ver_cadastros():
    exibir_cabecalho("PACIENTES CADASTRADOS")

    registros = atendimento_service.listar_cadastros()

    if not registros:
        print("\nNenhum paciente cadastrado.")
    else:
        for paciente in registros:
            (
                paciente_id, nome, idade,
                queixa, dor, status
            ) = paciente

            print(f"\nID: {paciente_id}")
            print(f"Nome: {nome}")
            print(f"Idade: {idade}")
            print(f"Queixa: {queixa}")
            print(f"Nível de dor: {dor}/10")
            print(f"Status: {status}")
            print("-" * 40)

    pausar()

             CENTRAL DE ATENDIMENTO - DR. COMMUNITY              

  [ 1 ] Fazer um novo cadastro
  [ 2 ] Fazer login na conta
  [ 3 ] Ver cadastros realizados
  [ 4 ] Sair do programa

-----------------------------------------------------------------

Digite o número da opção desejada: 1
                    NOVO CADASTRO DE PACIENTE                    

Qual é o seu nome completo? Isac de Oliveira Alexandre
Crie uma senha de acesso: 12345678
Qual é a sua idade? 23
Qual é a sua queixa/sintoma? dor de cabeça
Qual é o seu nível de dor (0 a 10)? 7

✅ Cadastro concluído com sucesso, Isac de Oliveira Alexandre!
Grau de gravidade: Urgente
